### chapter one: schwartzchild black holes

>what did one black hole say to the other black hole? you suck.

growing up, the explanations i got for black holes were always the same, 
> a black hole is an object whose gravity is so strong that not even light can escape

while that is technically correct, i think it's important to clarify that gravity isn't really acting like a "force pulling on photons" here.

perhaps the more technically correct explanation would be, 
> a black hole is a region of spacetime in which every future directed path points toward a singularity rather than toward the outside universe

all the paths converge, nothing leans outword, nothing is "held back"

and thus, we introduce einstein's field equations.

everything starts from, 
$$G_{\mu \nu} = \frac{8 \pi G}{c^4}T_{\mu \nu}$$

matter tells spacetime how to curve, curvature tells matter how to move.

and if enough mass-energy becomes concentrated into a sufficiently small region, the solution to these equations becomes the schwarzschild solution

the schwarzchild metric says that outside a non-rotating spherical mass,

$$ds^2 = -\left(1 - \frac{2M}{r}\right)dt^2 + \left(1 - \frac{2M}{r}\right)^{-1}dr^2 + r^2\left(d\theta^2 + \sin^2\theta \, d\phi^2\right)$$

describes the geometry of our black hole.

something really cool here is our $r$ value,
$$r_s = \frac{2GM}{c^2}$$
we call this the schwarzchild radius.

and this is the coolest spot in the black hole - the event horizon.


when,
$$ r = r_{s} $$

the metric coefficient $ 1 - \frac{2GM}{2c^2}$ goes to 0.

we used to think that this meant the maths "blew up" but really, it only means that the schwartzchild coordinates become bad coordinates (just like the latitude becomes singular at earth's poles).

if we change coordinates (say, to kruskal coordinates), this singularity complexity is removed.

the event horizon is not a physical surface.

so, the question is, what actually changes in the event horizon?

outside the horizon, 
* you can move inward or outward
* light cones point mostly upward in time

inside the horizon, the light cone tips inwards.

imagine spacetime itself rotating the allowed future directions.

outside,

      Future
        ^
       / \
      /   \
    
and inside, 

      \
       \
        \
         singularity

every possible future path points towards a smaller $r$. 

in fact, not even light cannot choose otherwise - not because gravity is stronger than light. more, becasuse "outwards" is no longer a future direction.

what a beautiful idea :3


a natural question that might come up at this point is, why can't light escape?

remember that photons always travel on null geodesics,
$$ds^2 = 0$$

outside the horizon, some null geodesics move outward.

inside the horizon, <b>every null geodesic moves towards a decreasing $r$

</b>there are literally no outgoing light rays.

so the question now is, what does this theory actually mean?

there's a famous quote by richard feynman:

> what i cannot create, i do not understand.

we can stare at the schwarzschild metric all day, but equations only become intuitive once we see what they actually do.

rather than treating general relativity as a collection of intimidating equations, we're going to build a black hole from the ground up and watch particles move through it.

our goal isn't just to make pretty animations (though, it mostly is), it's to build an intuition for how curved spacetime changes the possible paths that particles and light can take.

<b>so, let's simulate.</b>

before we start writing code, let's make one simplification. 

throughout this notebook, we'll be using geometrized units where,

$$ G = c = 1$$

if you are a quantum physicist who wants to come at me, you can go back to simulating photons or something.

In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# black hole mass (geometrized units)
M = 1.0
# schwarzschild radius (event horizon)            
R_S = 2 * M
# photon sphere        
R_PHOTON = 3 * M
# innermost stable circular orbit   
R_ISCO = 6 * M     

plt.rcParams['figure.dpi'] = 110
print(f"event horizon: r = {R_S}   photon sphere: r = {R_PHOTON}   ISCO: r = {R_ISCO}")

event horizon: r = 2.0   photon sphere: r = 3.0   ISCO: r = 6.0


In [2]:
def geodesic_rhs(tau, y, L, massive):
    """Right-hand side of the Schwarzschild geodesic equations (orbital plane)."""
    r, rdot, phi = y
    r2, r3, r4 = r*r, r**3, r**4
    acc = L*L / r3 - 3 * M * L*L / r4      # photon terms
    if massive:
        acc += -M / r2                      # Newtonian-like term for matter
    return [rdot, acc, L / r2]

def integrate_orbit(r0, rdot0, phi0, L, tau_max, massive=True, r_escape=200.0):
    """Integrate a geodesic; returns (r, phi, fate) with fate in {'captured','escaped','orbiting'}."""
    horizon = lambda t, y, *a: y[0] - (R_S * 1.001)
    horizon.terminal, horizon.direction = True, -1
    escape = lambda t, y, *a: y[0] - r_escape
    escape.terminal, escape.direction = True, 1

    sol = solve_ivp(geodesic_rhs, (0, tau_max), [r0, rdot0, phi0],
                    args=(L, massive), events=[horizon, escape],
                    rtol=1e-9, atol=1e-9, dense_output=True, max_step=1.0)
    tau = np.linspace(0, sol.t[-1], 4000)
    r, _, phi = sol.sol(tau)
    fate = 'captured' if sol.t_events[0].size else ('escaped' if sol.t_events[1].size else 'orbiting')
    return r, phi, fate

def energy_at_turning_point(r0, L):
    """Energy of a massive particle released at r0 with dr/dtau = 0."""
    return np.sqrt((1 - 2*M/r0) * (1 + L*L/(r0*r0)))

def to_3d(r, phi, inclination=0.0, node=0.0):
    """Place a planar orbit (r, phi) into 3D with a given inclination and line of nodes."""
    x, y = r*np.cos(phi), r*np.sin(phi)
    ci, si = np.cos(inclination), np.sin(inclination)
    cn, sn = np.cos(node), np.sin(node)
    X = cn*x - sn*(ci*y)
    Y = sn*x + cn*(ci*y)
    Z = si*y
    return X, Y, Z

In [ ]:
r = np.linspace(2.05, 40, 2000)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for L in [3.4, np.sqrt(12), 4.0, 4.5]:
    V = (1 - 2*M/r) * (1 + L**2/r**2)
    axes[0].plot(r, V, label=f"L = {L:.2f}")
axes[0].axvline(R_ISCO, color='gray', ls=':', label='ISCO (r = 6M)')
axes[0].set(title="Massive particles", xlabel="r / M", ylabel=r"$V_{\rm eff}$", ylim=(0.85, 1.1))
axes[0].legend(fontsize=8)

for L in [4.0, 3*np.sqrt(3), 6.0]:
    V = (1 - 2*M/r) * L**2/r**2
    axes[1].plot(r, V, label=f"L/E crit = {L:.2f}")
axes[1].axvline(R_PHOTON, color='gray', ls=':', label='Photon sphere (r = 3M)')
axes[1].set(title="Photons", xlabel="r / M", ylabel=r"$V_{\rm eff}$")
axes[1].legend(fontsize=8)

for ax in axes:
    ax.axvspan(0, R_S, color='k', alpha=0.25)
    ax.set_xlim(0, 40)
plt.tight_layout(); plt.show()

In [ ]:
def sphere(radius, n=40):
    u, v = np.mgrid[0:2*np.pi:n*1j, 0:np.pi:(n//2)*1j]
    return radius*np.cos(u)*np.sin(v), radius*np.sin(u)*np.sin(v), radius*np.cos(v)

orbits = []

# (a) Precessing bound orbit (start at apoapsis)
L = 3.9; r0 = 20.0
orbits.append((*integrate_orbit(r0, 0.0, 0.0, L, 3000, massive=True)[:2],
               0.0, 0.0, '#4fc3f7', 'Precessing bound orbit'))

# (b) Plunging orbit: low angular momentum -> capture
L = 3.2; r0 = 20.0
orbits.append((*integrate_orbit(r0, 0.0, 0.0, L, 3000, massive=True)[:2],
               np.radians(35), np.radians(60), '#ef5350', 'Plunge & capture'))

# (c) Zoom-whirl: energy just below the potential barrier peak
L = 4.2; r0 = 40.0
Vpeak_r = (L**2 - L*np.sqrt(L**2 - 12*M**2)) / (2*M)   # unstable circular radius
Vpeak = (1 - 2*M/Vpeak_r) * (1 + L**2/Vpeak_r**2)
E = np.sqrt(Vpeak) * 0.9995
rdot0 = -np.sqrt(max(E**2 - (1 - 2*M/r0)*(1 + L**2/r0**2), 0))
orbits.append((*integrate_orbit(r0, rdot0, 0.0, L, 4000, massive=True)[:2],
               np.radians(-25), np.radians(140), '#ab47bc', 'Zoom-whirl'))

# (d) Inclined circular orbit just outside the ISCO
r_c = 7.0; L = r_c*np.sqrt(M/(r_c - 3*M))              # circular-orbit angular momentum
orbits.append((*integrate_orbit(r_c, 0.0, 0.0, L, 1200, massive=True)[:2],
               np.radians(55), np.radians(200), '#66bb6a', 'Circular orbit (r = 7M)'))

fig = plt.figure(figsize=(11, 10))
ax = fig.add_subplot(111, projection='3d', facecolor='black')
fig.patch.set_facecolor('black')

Xs, Ys, Zs = sphere(R_S)
ax.plot_surface(Xs, Ys, Zs, color='black', edgecolor='#333', linewidth=0.2, shade=False, zorder=10)
Xp, Yp, Zp = sphere(R_PHOTON, n=24)
ax.plot_wireframe(Xp, Yp, Zp, color='#ff9800', linewidth=0.4, alpha=0.35)

for r_arr, phi_arr, inc, node, color, label in orbits:
    X, Y, Z = to_3d(r_arr, phi_arr, inc, node)
    ax.plot(X, Y, Z, color=color, lw=1.1, label=label)
    ax.scatter(X[-1], Y[-1], Z[-1], color=color, s=25)

lim = 22
ax.set(xlim=(-lim, lim), ylim=(-lim, lim), zlim=(-lim, lim))
ax.set_box_aspect((1, 1, 1)); ax.set_axis_off()
ax.legend(loc='upper left', facecolor='black', labelcolor='white', fontsize=9)
ax.set_title("Geodesics around a Schwarzschild black hole", color='white')
plt.tight_layout(); plt.show()

In [ ]:
def photon_ray(b, D=60.0, lam_max=400.0):
    """Photon launched from x = -D travelling in +x with impact parameter b (E = 1, L = b)."""
    r0 = np.hypot(D, b)
    phi0 = np.arctan2(b, -D)
    rdot0 = -np.sqrt(max(1.0 - (1 - 2*M/r0) * b*b / (r0*r0), 0.0))
    return integrate_orbit(r0, rdot0, phi0, b, lam_max, massive=False, r_escape=1.5*D)

b_crit = 3*np.sqrt(3)*M
impact_params = np.concatenate([np.linspace(2.0, 4.8, 5),
                                [b_crit*0.999, b_crit*1.001, b_crit*1.01],
                                np.linspace(5.6, 12, 6)])

fig = plt.figure(figsize=(11, 9))
ax = fig.add_subplot(111, projection='3d', facecolor='black')
fig.patch.set_facecolor('black')

Xs, Ys, Zs = sphere(R_S)
ax.plot_surface(Xs, Ys, Zs, color='black', edgecolor='#333', linewidth=0.2, shade=False)
Xp, Yp, Zp = sphere(R_PHOTON, n=24)
ax.plot_wireframe(Xp, Yp, Zp, color='#ff9800', linewidth=0.35, alpha=0.3)

for i, b in enumerate(impact_params):
    r_arr, phi_arr, fate = photon_ray(b)
    # spread the rays around the axis of approach so the bundle is genuinely 3D
    X, Y, Z = to_3d(r_arr, phi_arr, inclination=0, node=0)
    angle = i * 2*np.pi / len(impact_params)
    Yr, Zr = Y*np.cos(angle), Y*np.sin(angle)
    color = '#ef5350' if fate == 'captured' else ('#ffee58' if b < 5.5 else '#4fc3f7')
    ax.plot(X, Yr, Zr, color=color, lw=0.9, alpha=0.9)

lim = 30
ax.set(xlim=(-lim, lim), ylim=(-lim, lim), zlim=(-lim, lim))
ax.set_box_aspect((1, 1, 1)); ax.set_axis_off()
ax.set_title("Gravitational lensing: light rays with different impact parameters\n"
             "red = captured, yellow = near-critical whirl, blue = deflected", color='white')
plt.tight_layout(); plt.show()

In [ ]:
r = np.linspace(R_S, 30, 300)
theta = np.linspace(0, 2*np.pi, 120)
Rg, Tg = np.meshgrid(r, theta)
Zg = -2*np.sqrt(2*M*(Rg - R_S))          # flip downward: the "well"
Xg, Yg = Rg*np.cos(Tg), Rg*np.sin(Tg)

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d', facecolor='black')
fig.patch.set_facecolor('black')
ax.plot_surface(Xg, Yg, Zg, cmap='inferno', alpha=0.75, linewidth=0, antialiased=True)

# drape the precessing orbit on the surface
r_o, phi_o, _ = integrate_orbit(20.0, 0.0, 0.0, 3.9, 2500, massive=True)
mask = r_o > R_S + 1e-3
z_o = -2*np.sqrt(2*M*(r_o[mask] - R_S))
ax.plot(r_o[mask]*np.cos(phi_o[mask]), r_o[mask]*np.sin(phi_o[mask]), z_o + 0.15,
        color='#4fc3f7', lw=1.2, label='Precessing orbit')

ax.set_box_aspect((1, 1, 0.5)); ax.set_axis_off()
ax.legend(facecolor='black', labelcolor='white')
ax.set_title("Flamm's paraboloid: the curvature of space around the black hole", color='white')
ax.view_init(elev=28, azim=-60)
plt.tight_layout(); plt.show()

In [ ]:
from matplotlib import animation
from IPython.display import HTML

r1, p1, _ = integrate_orbit(20.0, 0.0, 0.0, 3.9, 2200, massive=True)
r2, p2, _ = integrate_orbit(20.0, 0.0, np.pi, 3.2, 2200, massive=True)
X1, Y1, Z1 = to_3d(r1, p1, np.radians(10), 0)
X2, Y2, Z2 = to_3d(r2, p2, np.radians(-20), np.radians(90))

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d', facecolor='black')
fig.patch.set_facecolor('black')
Xs, Ys, Zs = sphere(R_S)
ax.plot_surface(Xs, Ys, Zs, color='black', edgecolor='#333', linewidth=0.2, shade=False)
lim = 22
ax.set(xlim=(-lim, lim), ylim=(-lim, lim), zlim=(-lim, lim))
ax.set_box_aspect((1, 1, 1)); ax.set_axis_off()

trail1, = ax.plot([], [], [], color='#4fc3f7', lw=1.0)
trail2, = ax.plot([], [], [], color='#ef5350', lw=1.0)
dot1, = ax.plot([], [], [], 'o', color='#4fc3f7', ms=5)
dot2, = ax.plot([], [], [], 'o', color='#ef5350', ms=5)

n_frames, step = 150, len(X1)//150
def update(f):
    i = min((f+1)*step, len(X1)-1)
    trail1.set_data_3d(X1[:i], Y1[:i], Z1[:i]); dot1.set_data_3d([X1[i]], [Y1[i]], [Z1[i]])
    j = min(i, len(X2)-1)
    trail2.set_data_3d(X2[:j], Y2[:j], Z2[:j]); dot2.set_data_3d([X2[j]], [Y2[j]], [Z2[j]])
    ax.view_init(elev=22, azim=f*0.8 - 60)
    return trail1, trail2, dot1, dot2

anim = animation.FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())